# Explainable Industrial Defect Inspector — Colab Setup

Mounts Drive, clones the repo, installs dependencies, and creates the project folder structure.

**Workflow:** code locally → push to GitHub → run this notebook to test.

- Data lives in Drive: `defect_inspector/data/mvtec`
- Results live in Drive: `defect_inspector/outputs/`
- Code is cloned from GitHub to `/content/xAI_Defects`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/defect_inspector"
REPO_URL  = "https://github.com/AKIF-jk/xAI_Defects.git"
LOCAL_DIR = "/content/xAI_Defects"

# Mount point for Drive data
DRIVE_DATA    = os.path.join(DRIVE_ROOT, "data")
DRIVE_OUTPUTS = os.path.join(DRIVE_ROOT, "outputs")

os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"Drive root: {DRIVE_ROOT}")
print(f"Exists: {os.path.exists(DRIVE_ROOT)}")

In [ ]:
%cd /content
if os.path.exists(LOCAL_DIR):
    %cd {LOCAL_DIR}
    !git pull
else:
    !git clone {REPO_URL}
    %cd {LOCAL_DIR}

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --quiet
!pip install open_clip_torch timm captum shap albumentations fastapi uvicorn python-multipart gradio faiss-cpu anthropic matplotlib seaborn scikit-learn --quiet

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name:      {torch.cuda.get_device_name(0)}")
    print(f"GPU count:     {torch.cuda.device_count()}")

In [ ]:
import open_clip, captum, shap, gradio
print(f"torch:              {torch.__version__}")
print(f"open_clip_torch:    {open_clip.__version__}")
print(f"captum:             {captum.__version__}")
print(f"shap:               {shap.__version__}")
print(f"gradio:             {gradio.__version__}")

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/defect_inspector"

folders = [
    "data/mvtec",
    "src/data", "src/model", "src/xai", "src/api",
    "outputs/heatmaps", "outputs/shap", "outputs/results",
]

for f in folders:
    os.makedirs(os.path.join(DRIVE_ROOT, f), exist_ok=True)

print("Drive folder structure created:")
for f in folders:
    path = os.path.join(DRIVE_ROOT, f)
    print(f"  {path}")

In [ ]:
import sys
SRC_PATH = os.path.join(DRIVE_ROOT, "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

LOCAL_SRC = "/content/xAI_Defects/src"
if os.path.exists(LOCAL_SRC) and LOCAL_SRC not in sys.path:
    sys.path.insert(0, LOCAL_SRC)

print(f"sys.path entries:\n  {SRC_PATH}\n  {LOCAL_SRC}")

In [ ]:
print("Environment ready")

## Part 2: MVTec AD Dataset

Download, verify, and visualize the 15-category benchmark.

Uses kagglehub (no API token needed for public datasets). Data is copied to Drive for persistence across Colab sessions.

In [ ]:
!pip install kagglehub -q

import os
import kagglehub

MVTEC_DIR = "/content/drive/MyDrive/defect_inspector/data/mvtec"
EXTRACTED = os.path.join(MVTEC_DIR, "mvtec_anomaly_detection")

if os.path.exists(EXTRACTED):
    print(f"MVTec AD already in Drive: {EXTRACTED}")
else:
    os.makedirs(MVTEC_DIR, exist_ok=True)
    print("Downloading MVTec AD from Kaggle...")
    cache_path = kagglehub.dataset_download("ipythonx/mvtec-ad")
    print(f"Cached at: {cache_path}")

    contents = os.listdir(cache_path)
    root_dirs = [d for d in contents if os.path.isdir(os.path.join(cache_path, d))]

    if len(root_dirs) == 1:
        !cp -r "{cache_path}/{root_dirs[0]}" "{EXTRACTED}"
    else:
        !cp -r "{cache_path}" "{EXTRACTED}"

    print(f"MVTec AD ready at: {EXTRACTED}")

In [ ]:
import os, glob

EXTRACTED = "/content/drive/MyDrive/defect_inspector/data/mvtec/mvtec_anomaly_detection"

CATEGORIES = ["bottle","cable","capsule","carpet","grid","hazelnut","leather",
              "metal_nut","pill","screw","tile","toothbrush","transistor","wood","zipper"]

found = [c for c in CATEGORIES if os.path.isdir(os.path.join(EXTRACTED, c))]
missing = [c for c in CATEGORIES if c not in found]
print(f"Categories found: {len(found)}/15")
if missing:
    print(f"Missing: {missing}")

print(f"\n{'Category':<15} {'Train':>6} {'Test':>6} {'Defect Types'}")
print("-" * 65)

category_stats = {}
for cat in CATEGORIES:
    cat_dir = os.path.join(EXTRACTED, cat)
    if not os.path.isdir(cat_dir):
        continue

    train_dir = os.path.join(cat_dir, "train", "good")
    train_count = len(glob.glob(os.path.join(train_dir, "*.png")))

    test_dir = os.path.join(cat_dir, "test")
    test_dirs = [d for d in os.listdir(test_dir) if os.path.isdir(os.path.join(test_dir, d))]
    defect_types = sorted(d for d in test_dirs if d != "good")
    test_anom = sum(len(glob.glob(os.path.join(test_dir, d, "*.png"))) for d in defect_types)
    test_good = len(glob.glob(os.path.join(test_dir, "good", "*.png")))
    test_count = test_good + test_anom

    print(f"{cat:<15} {train_count:>6} {test_count:>6} {', '.join(defect_types)}")
    category_stats[cat] = {"train": train_count, "normal_test": test_good,
                          "anom_test": test_anom, "defects": defect_types}

In [ ]:
import matplotlib.pyplot as plt

cats = list(category_stats.keys())
anom_counts = [category_stats[c]["anom_test"] for c in cats]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(cats)), anom_counts, color="crimson", edgecolor="black", linewidth=0.5)
ax.set_xticks(range(len(cats)))
ax.set_xticklabels(cats, rotation=45, ha="right")
ax.set_ylabel("Anomalous Test Images")
ax.set_title("MVTec AD: Anomalous Test Images per Category")
ax.grid(axis="y", alpha=0.3)

for bar, count in zip(bars, anom_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(count), ha="center", va="bottom", fontsize=9)

plt.tight_layout()

save_path = "/content/drive/MyDrive/defect_inspector/outputs/results/dataset_overview.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {save_path}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

EXTRACTED = "/content/drive/MyDrive/defect_inspector/data/mvtec/mvtec_anomaly_detection"
bottle_dir = os.path.join(EXTRACTED, "bottle")

test_dir = os.path.join(bottle_dir, "test")
defect_dirs = sorted(d for d in os.listdir(test_dir)
                     if d != "good" and os.path.isdir(os.path.join(test_dir, d)))
defect_type = defect_dirs[0]

normal_path = os.path.join(bottle_dir, "train", "good",
                           sorted(os.listdir(os.path.join(bottle_dir, "train", "good")))[0])
anom_path = os.path.join(bottle_dir, "test", defect_type,
                         sorted(os.listdir(os.path.join(bottle_dir, "test", defect_type)))[0])
mask_path = os.path.join(bottle_dir, "ground_truth", defect_type,
                         sorted(os.listdir(os.path.join(bottle_dir, "ground_truth", defect_type)))[0])

normal = plt.imread(normal_path)
anomalous = plt.imread(anom_path)
mask = plt.imread(mask_path)

if mask.ndim == 3:
    mask_gray = mask[:, :, 0]
else:
    mask_gray = mask

overlay = np.zeros((*mask_gray.shape, 4))
overlay[:, :, 0] = 1.0
overlay[:, :, 3] = mask_gray * 0.6

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(normal)
axes[0].set_title("Normal (good)")
axes[0].axis("off")

axes[1].imshow(anomalous)
axes[1].set_title(f"Anomalous ({defect_type})")
axes[1].axis("off")

axes[2].imshow(anomalous)
axes[2].imshow(overlay)
axes[2].set_title("Anomalous + Defect Mask")
axes[2].axis("off")

plt.tight_layout()
plt.show()
print(f"Normal: {normal_path}\nAnomalous: {anom_path}\nMask: {mask_path}")

## Part 3: Validate MVTecDataset Module

Instantiates the dataset, prints per-category stats, verifies tensor shapes, and visualises a 3x4 random grid with mask overlays.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import transforms

sys.path.insert(0, "/content/xAI_Defects/src")
from data.mvtec_dataset import MVTecDataset, get_dataloaders

EXTRACTED = "/content/drive/MyDrive/defect_inspector/data/mvtec/mvtec_anomaly_detection"

tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
mtf = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor(),
])

train_ds = MVTecDataset(EXTRACTED, "bottle", split="train")
test_ds  = MVTecDataset(EXTRACTED, "bottle", split="test",
                        transform=tf, mask_transform=mtf)

train_count = len(train_ds)
test_count = len(test_ds)
anom_count = sum(1 for i in range(test_count) if test_ds.labels[i] == 1)
normal_count = test_count - anom_count

print(f"Bottle dataset statistics:")
print(f"  Total train images:  {train_count}")
print(f"  Total test images:   {test_count}")
print(f"  Anomalous test:      {anom_count}")
print(f"  Normal test:         {normal_count}")

img, mask, label = test_ds[0]
print(f"\nTensor shapes:")
print(f"  Image: {img.shape}")
print(f"  Mask:  {mask.shape}")
assert img.shape == (3, 256, 256), f"Expected (3,256,256), got {img.shape}"
assert mask.shape == (1, 256, 256), f"Expected (1,256,256), got {mask.shape}"
print("Shape assertions passed.")

indices = np.random.choice(len(test_ds), 12, replace=False)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

for ax, idx in zip(axes.flat, indices):
    img, mask, label = test_ds[idx]

    img_vis = img * std + mean
    img_vis = torch.clamp(img_vis, 0, 1)
    img_np = img_vis.permute(1, 2, 0).numpy()

    mask_np = mask.squeeze(0).numpy()

    overlay = img_np.copy()
    overlay[:, :, 0] = np.clip(overlay[:, :, 0] + mask_np * 0.5, 0, 1)

    ax.imshow(overlay)
    ax.set_title("Anomalous" if label.item() == 1 else "Normal", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()
print("MVTecDataset module validation complete.")